In [1]:
import torch
import torch.nn as nn
import torch.autograd as autograd
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, utils
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
# Base directories (your cwd is C:\Users\ADVAIT\Desktop\CV CASE 2)
BASE_DIR  = Path.cwd()
DATA_DIR  = BASE_DIR / "final_processed_data" / "final_processed_data"
TRAIN_DIR = DATA_DIR / "train"

print("Using training folder:", TRAIN_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Using training folder: C:\Users\ADVAIT\Desktop\CV CASE 2\final_processed_data\final_processed_data\train


In [3]:
# 2) DataLoader
img_size = 64
batch_size = 64

tf = transforms.Compose([
    transforms.Resize(img_size),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3),
])

train_ds = datasets.ImageFolder(root=str(TRAIN_DIR), transform=tf)
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)

print("Classes:", train_ds.classes)



Classes: ['drink', 'food', 'inside', 'menu', 'outside']


In [4]:
# 3) WGAN-GP Generator & Critic
class Generator(nn.Module):
    def __init__(self, z_dim=100, img_channels=3, feature_g=64):
        super().__init__()
        self.net = nn.Sequential(
            # input: N x z_dim
            nn.Linear(z_dim, feature_g*8*4*4),
            nn.ReLU(True),
            nn.Unflatten(1, (feature_g*8, 4, 4)),
            # upsample to 8×8
            nn.ConvTranspose2d(feature_g*8, feature_g*4, 4, 2, 1), nn.ReLU(True),
            # upsample to 16×16
            nn.ConvTranspose2d(feature_g*4, feature_g*2, 4, 2, 1), nn.ReLU(True),
            # upsample to 32×32
            nn.ConvTranspose2d(feature_g*2, feature_g,   4, 2, 1), nn.ReLU(True),
            # upsample to 64×64
            nn.ConvTranspose2d(feature_g, img_channels, 4, 2, 1), nn.Tanh()
        )
    def forward(self, z):
        return self.net(z)

class Critic(nn.Module):
    def __init__(self, img_channels=3, feature_d=64):
        super().__init__()
        self.net = nn.Sequential(
            # input: N x 3 x64x64
            nn.Conv2d(img_channels, feature_d, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d,   feature_d*2, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d*2, feature_d*4, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d*4, feature_d*8, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(feature_d*8*4*4, 1),
        )
    def forward(self, img):
        return self.net(img)


In [5]:
# 4) Gradient Penalty
def gradient_penalty(critic, real, fake, device):
    batch_size = real.size(0)
    # sample interpolation
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    # critic score
    crit_interp = critic(interp)
    # compute gradients w.r.t. interpolated images
    grads = autograd.grad(
        outputs=crit_interp,
        inputs=interp,
        grad_outputs=torch.ones_like(crit_interp),
        create_graph=True
    )[0]
    grads = grads.view(batch_size, -1)
    gp = ((grads.norm(2, dim=1) - 1) ** 2).mean()
    return gp


In [15]:
# 5) Training Loop (updated with device variable and model saving)

def train_wgangp(G, C, dl, z_dim, n_epochs, lr, device, 
                 critic_iters=5, lambda_gp=10):
    G, C = G.to(device), C.to(device)
    optG = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.0, 0.9))
    optC = torch.optim.Adam(C.parameters(), lr=lr, betas=(0.0, 0.9))

    for epoch in range(1, n_epochs+1):
        for i, (real_imgs, _) in enumerate(dl):
            real_imgs = real_imgs.to(device)
            bsz = real_imgs.size(0)

            # --- Train Critic ---
            for _ in range(critic_iters):
                z = torch.randn(bsz, z_dim, device=device)
                fake_imgs = G(z).detach()
                real_scores = C(real_imgs)
                fake_scores = C(fake_imgs)
                gp = gradient_penalty(C, real_imgs, fake_imgs, device)
                c_loss = fake_scores.mean() - real_scores.mean() + lambda_gp * gp

                optC.zero_grad()
                c_loss.backward()
                optC.step()

            # --- Train Generator ---
            z = torch.randn(bsz, z_dim, device=device)
            gen_imgs = G(z)
            g_loss = -C(gen_imgs).mean()

            optG.zero_grad()
            g_loss.backward()
            optG.step()

        print(f"Epoch {epoch}/{n_epochs}  C_loss: {c_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    # === Save models after training ===
    torch.save(G.state_dict(), "wgangp_generator.pth")
    torch.save(C.state_dict(), "wgangp_critic.pth")
    print("Models saved: wgangp_generator.pth, wgangp_critic.pth")

    return G, C

# Hyperparameters
z_dim = 100
lr    = 1e-4
epochs = 30

# Instantiate models
G = Generator(z_dim=z_dim)
C = Critic()

# Train
G, C = train_wgangp(G, C, train_dl, z_dim, epochs, lr, device=device)


Epoch 1/30  C_loss: -106.5287  G_loss: -82.5711
Epoch 2/30  C_loss: -64.9320  G_loss: -88.6331
Epoch 3/30  C_loss: -55.8095  G_loss: -147.3204
Epoch 4/30  C_loss: -42.6189  G_loss: -90.2959
Epoch 5/30  C_loss: -37.1307  G_loss: -116.6199
Epoch 6/30  C_loss: -37.1119  G_loss: -76.0010
Epoch 7/30  C_loss: -33.0431  G_loss: -81.2893
Epoch 8/30  C_loss: -33.1813  G_loss: -65.6676
Epoch 9/30  C_loss: -31.3743  G_loss: -60.7274
Epoch 10/30  C_loss: -26.7726  G_loss: -49.0600
Epoch 11/30  C_loss: -21.2510  G_loss: -33.2356
Epoch 12/30  C_loss: -27.0385  G_loss: -36.7705
Epoch 13/30  C_loss: -49.0165  G_loss: -55.8248
Epoch 14/30  C_loss: -28.8574  G_loss: 10.8448
Epoch 15/30  C_loss: -23.9967  G_loss: -23.3363
Epoch 16/30  C_loss: -22.7623  G_loss: -18.9434
Epoch 17/30  C_loss: -24.4367  G_loss: -18.0391
Epoch 18/30  C_loss: -26.4639  G_loss: 23.2569
Epoch 19/30  C_loss: -35.9585  G_loss: 22.3425
Epoch 20/30  C_loss: -24.2876  G_loss: 1.1018
Epoch 21/30  C_loss: -17.9088  G_loss: -11.8048
Epo

In [6]:
import torch

# Instantiate the Generator (same z_dim as before)
G = Generator(z_dim=100)
G.load_state_dict(torch.load("wgangp_generator.pth", map_location=device))
G = G.to(device)
G.eval()


Generator(
  (net): Sequential(
    (0): Linear(in_features=100, out_features=8192, bias=True)
    (1): ReLU(inplace=True)
    (2): Unflatten(dim=1, unflattened_size=(512, 4, 4))
    (3): ConvTranspose2d(512, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): ConvTranspose2d(64, 3, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (10): Tanh()
  )
)

In [7]:
import torch

print(torch.cuda.is_available())  # True or False?
print(device)                     # Should be 'cuda' or 'cpu'
print(G.__class__)                # Should print <class 'Generator'>


True
cuda
<class '__main__.Generator'>


In [8]:
device = torch.device("cpu")
G = Generator(z_dim=100)
G.load_state_dict(torch.load("wgangp_generator.pth", map_location="cpu"))
G.eval()

with torch.no_grad():
    z = torch.randn(1, 100)
    sample = G(z)
print("Sample shape:", sample.shape)  # Should be [1, 3, 64, 64]


Sample shape: torch.Size([1, 3, 64, 64])


In [11]:
from PIL import Image
import numpy as np

G = Generator(z_dim=100)
G.load_state_dict(torch.load("wgangp_generator.pth", map_location="cpu"))
G.eval()

for i in range(5):
    with torch.no_grad():
        z = torch.randn(1, 100)
        sample = G(z)
        img = sample[0].detach().cpu().numpy()
        img = np.transpose(img, (1, 2, 0))
        img = (img * 0.5) + 0.5
        img = np.clip(img, 0, 1)
        img_uint8 = (img * 255).astype(np.uint8)
        Image.fromarray(img_uint8).save(f"sample_{i+1}.png")
        print(f"Saved sample_{i+1}.png")


Saved sample_1.png
Saved sample_2.png
Saved sample_3.png
Saved sample_4.png
Saved sample_5.png


Chat gpt first prompt: which libraries will be use in order to train wgan_gp model.
Chat gpt last prompt: how can I improve image quality that we had saved now.